In [1]:
import hashlib
import geopandas as gpd

In [2]:
def gen_hash(s: str):
    return hashlib.md5(s.encode()).digest().hex()

In [3]:
PATH_VECTORS = "/mnt/data/workspace/bogor-agrisat/data/bogor-vectors"
PATH_AOI = "/mnt/data/workspace/bogor-agrisat/data/production-data/area-of-interest"
PATH_ATTRIBUTES = "/mnt/data/workspace/bogor-agrisat/data/production-data/attributes"

## Bogor

In [4]:
df_extent = gpd.read_file(f"{PATH_VECTORS}/bogor-kecamatan.shp")
df_extent.head()

,Shape_Leng,Shape_Area,ADM3_EN,ADM3_PCODE,ADM3_REF,ADM3ALT1EN,ADM3ALT2EN,ADM2_EN,ADM2_PCODE,ADM1_EN,ADM1_PCODE,ADM0_EN,ADM0_PCODE,date,validOn,validTo,geometry
0,0.679195,0.007535,Babakan Madang,ID3201140,None,None,None,Bogor,ID3201,Jawa Barat,ID32,Indonesia,ID,2019-12-20,2020-04-01,NaT,"POLYGON ((704623.815 9280656.007, 704647.464 9..."
1,0.364427,0.001862,Bogor Barat,ID3271050,None,None,None,Kota Bogor,ID3271,Jawa Barat,ID32,Indonesia,ID,2019-12-20,2020-04-01,NaT,"POLYGON ((695147.82 9276778.316, 695162.095 92..."
2,0.341327,0.002514,Bogor Selatan,ID3271010,None,None,None,Kota Bogor,ID3271,Jawa Barat,ID32,Indonesia,ID,2019-12-20,2020-04-01,NaT,"POLYGON ((698573.372 9269342.909, 698609.187 9..."
3,0.184981,0.000654,Bogor Tengah,ID3271040,None,None,None,Kota Bogor,ID3271,Jawa Barat,ID32,Indonesia,ID,2019-12-20,2020-04-01,NaT,"POLYGON ((697749.155 9273044.758, 697749.249 9..."
4,0.194338,0.000867,Bogor Timur,ID3271020,None,None,None,Kota Bogor,ID3271,Jawa Barat,ID32,Indonesia,ID,2019-12-20,2020-04-01,NaT,"POLYGON ((702489.833 9269511.729, 702492.124 9..."


### Kecamatan

In [5]:
df_kec = df_extent.copy()

df_kec["hash"] = df_kec["ADM3_PCODE"].apply(gen_hash)
df_kec["area"] = df_kec.area
df_kec["level"] = "subdistrict"

df_kec = df_kec.rename(columns={"ADM3_EN": "name", "ADM2_EN": "city"})
df_kec = df_kec[["hash", "level", "name", "city", "area", "geometry"]]

df_kec.head()

,hash,level,name,city,area,geometry
0,024cae77e8160a5b5e93d5233792a44d,subdistrict,Babakan Madang,Bogor,9.218301e+07,"POLYGON ((704623.815 9280656.007, 704647.464 9..."
1,8cadd023ed86500ab59783cfd590ef5d,subdistrict,Bogor Barat,Kota Bogor,2.278099e+07,"POLYGON ((695147.82 9276778.316, 695162.095 92..."
2,aa9beadb27fa63e7954913ac9cdd8b98,subdistrict,Bogor Selatan,Kota Bogor,3.075366e+07,"POLYGON ((698573.372 9269342.909, 698609.187 9..."
3,f4d64f05d6c8b462f72d9bd139a67dd3,subdistrict,Bogor Tengah,Kota Bogor,8.000876e+06,"POLYGON ((697749.155 9273044.758, 697749.249 9..."
4,debe0272a16f9eb51b51f6b860865911,subdistrict,Bogor Timur,Kota Bogor,1.060660e+07,"POLYGON ((702489.833 9269511.729, 702492.124 9..."


In [ ]:
df_kec.to_file(f"{PATH_AOI}/bogor-kecamatan.geojson")

### Kota

In [7]:
df_kota = df_extent.dissolve(by="ADM2_PCODE").reset_index()

df_kota["hash"] = df_kota["ADM2_PCODE"].apply(gen_hash)
df_kota["area"] = df_kota.area
df_kota["level"] = "city"
df_kota["name"] = df_kota["ADM2_EN"]

df_kota = df_kota.rename(columns={"ADM2_EN": "city"})
df_kota = df_kota[["hash", "level", "name", "city", "area", "geometry"]]

df_kota.head()

,hash,level,name,city,area,geometry
0,e1a86a89fb5823ba996fb036f4bfe0c3,city,Bogor,Bogor,2.993798e+09,"POLYGON ((692269.408 9257374.102, 692122.53 92..."
1,50711b04bdda08f99f22cacbe6cf8be7,city,Kota Bogor,Kota Bogor,1.117602e+08,"POLYGON ((704240.674 9263981.972, 704239.252 9..."


In [ ]:
df_kota.to_file(f"{PATH_AOI}/bogor-kota.geojson")

### Bogor Extent

In [9]:
df_bogor = df_extent.dissolve().reset_index()

df_bogor["hash"] = gen_hash("bogor-extent")
df_bogor["area"] = df_bogor.area
df_bogor["level"] = "extent"
df_bogor["name"] = "Bogor"
df_bogor["city"] = "Bogor"

df_bogor = df_bogor[["hash", "level", "name", "city", "area", "geometry"]]

df_bogor.head()

,hash,level,name,city,area,geometry
0,30fcd6e95eae1ef724d76ab17da9afe5,extent,Bogor,Bogor,3.105558e+09,"POLYGON ((711283.635 9249268.012, 711141.74 92..."


In [ ]:
df_bogor.to_file(f"{PATH_AOI}/bogor-extent.geojson")

## Sawah

In [11]:
df_sawah_raw = gpd.read_file(f"{PATH_VECTORS}/bogor-sawah.shp")
df_sawah_raw.head()

,id,Name,descriptio,timestamp,begin,end,altitudeMo,tessellate,extrude,visibility,drawOrder,icon,geometry
0,0F78A7C572394311EC28,Sawah Fahmi 1,None,None,None,None,None,-1,0,-1,NaN,None,"POLYGON ((681307.968 9270039.047, 681304.858 9..."
1,077CEB8F2F3EEE828DFE,Sawah Fahmi 2,None,None,None,None,None,-1,0,-1,NaN,None,"POLYGON ((681310.63 9270011.994, 681320.426 92..."
2,050E432C013EEE841178,Sawah 1,None,None,None,None,None,-1,0,-1,NaN,None,"POLYGON ((681324.771 9270004.296, 681339.943 9..."
3,06460832AA3EEE84D880,Sawah 2,None,None,None,None,None,-1,0,-1,NaN,None,"POLYGON ((681239.907 9270070.376, 681247.891 9..."
4,02A98ED9B03EEE85B45C,Sawah 3,None,None,None,None,None,-1,0,-1,NaN,None,"POLYGON ((681230.418 9269981.876, 681235.536 9..."


In [12]:
df_sawah = df_sawah_raw.copy()

df_sawah["hash"] = df_sawah["id"].apply(gen_hash)
df_sawah["area"] = df_sawah.area
df_sawah["level"] = "rice_paddy"
df_sawah["city"] = "Bogor"

df_sawah = df_sawah.rename(columns={"Name": "name"})
df_sawah = df_sawah[["hash", "level", "name", "city", "area", "geometry"]]

df_sawah.head()

,hash,level,name,city,area,geometry
0,f698b891c432a8fc9aeebd8cfaca9c51,rice_paddy,Sawah Fahmi 1,Bogor,1276.349585,"POLYGON ((681307.968 9270039.047, 681304.858 9..."
1,92eac953eef8b4ce4e0aac12f1df207f,rice_paddy,Sawah Fahmi 2,Bogor,282.008418,"POLYGON ((681310.63 9270011.994, 681320.426 92..."
2,60307db86871b782f5bc1d5d88a723b3,rice_paddy,Sawah 1,Bogor,6163.396308,"POLYGON ((681324.771 9270004.296, 681339.943 9..."
3,aa3f6bfc857119446857286da1ab42e2,rice_paddy,Sawah 2,Bogor,2618.041895,"POLYGON ((681239.907 9270070.376, 681247.891 9..."
4,c7c41c1b295fab2fc616f01fc4e7e0db,rice_paddy,Sawah 3,Bogor,1075.352897,"POLYGON ((681230.418 9269981.876, 681235.536 9..."


In [ ]:
df_sawah.to_file(f"{PATH_AOI}/bogor-sawah.geojson")